# Generate DS1

This notebook creates the public synthetic stand-in dataset used by `reproduce_paper.ipynb`.

Output file: `data/DS1.xlsx`.

## 1. Configuration

The workbook contains only the columns needed by the reproduction workflow: 14 chemistry variables, `IMS`, `Oven Recipe`, measured targets, and cached NaMo-like estimates.

In [7]:
from pathlib import Path

import numpy as np
import pandas as pd

REPO_DIR = Path.cwd().resolve()
OUTPUT_DIR = REPO_DIR / "data"
OUTPUT_DIR.mkdir(exist_ok=True)

DS1_FILE = OUTPUT_DIR / "DS1.xlsx"
NAMO_FILE = OUTPUT_DIR / "NaMo_syntetic.xlsx"

SEED = 52
TOTAL_ROWS = 9000
rng = np.random.default_rng(SEED)

## 2. Synthetic Design

The generator uses broad, hand-set alloy-family and aging-recipe effects. This creates visible dependencies among chemistry, recipe, NaMo-like estimates, and tensile properties without copying statistical details from the original industrial dataset.

In [8]:
IMS_OPTIONS = ['6060', '6063', '6005 v1', '6005 v2', '6082 v1', '6082 v2', '6082 v3']
IMS_PROBABILITIES = np.array([0.16, 0.14, 0.15, 0.13, 0.15, 0.14, 0.13])
RECIPE_OPTIONS = [
    "Oven Recipe 1",
    "Oven Recipe 2",
    "Oven Recipe 3",
    "Oven Recipe 4",
    "Oven Recipe 5",
    "Oven Recipe 6",
]

STRENGTH_CODE = {
    '6060': 0.0,
    '6063': 0.3,
    '6005 v1': 0.8,
    '6005 v2': 1.1,
    '6082 v1': 1.5,
    '6082 v2': 1.7,
    '6082 v3': 2.0,
}

RECIPE_HEAT = {
    "Oven Recipe 1": 0.6,
    "Oven Recipe 2": 0.2,
    "Oven Recipe 3": -0.6,
    "Oven Recipe 4": 0.5,
    "Oven Recipe 5": -0.8,
    "Oven Recipe 6": 0.1,
}

RECIPE_STRENGTH = {
    "Oven Recipe 1": 18,
    "Oven Recipe 2": 8,
    "Oven Recipe 3": -10,
    "Oven Recipe 4": 15,
    "Oven Recipe 5": -16,
    "Oven Recipe 6": 4,
}

RECIPE_WEIGHTS = {
    '6060': np.array([0.10, 0.18, 0.22, 0.10, 0.28, 0.12]),
    '6063': np.array([0.14, 0.18, 0.14, 0.14, 0.20, 0.20]),
    '6005 v1': np.array([0.20, 0.24, 0.08, 0.18, 0.18, 0.12]),
    '6005 v2': np.array([0.22, 0.16, 0.10, 0.20, 0.18, 0.14]),
    '6082 v1': np.array([0.16, 0.14, 0.10, 0.22, 0.20, 0.18]),
    '6082 v2': np.array([0.18, 0.12, 0.08, 0.24, 0.18, 0.20]),
    '6082 v3': np.array([0.12, 0.12, 0.10, 0.20, 0.22, 0.24]),
}

def clip(values, low, high):
    return np.clip(values, low, high)


## 3. Generate the Minimal Table

The NaMo-like columns are deliberately imperfect and biased compared with the generated measured targets. That keeps the synthetic results meaningful for comparing NaMo, XGBoost, hybrid, and baseline behavior without encoding original confidential statistics.

In [15]:
ims = rng.choice(IMS_OPTIONS, size=TOTAL_ROWS, p=IMS_PROBABILITIES)
recipes = np.array([rng.choice(RECIPE_OPTIONS, p=RECIPE_WEIGHTS[value]) for value in ims])
strength = pd.Series(ims).map(STRENGTH_CODE).to_numpy()
heat = pd.Series(recipes).map(RECIPE_HEAT).to_numpy()
batch = rng.normal(0, 1, TOTAL_ROWS)
process = rng.normal(0, 1, TOTAL_ROWS)

Mg = clip(0.34 + 0.17 * strength + 0.04 * batch + rng.normal(0, 0.028, TOTAL_ROWS), 0.30, 1.15)
Si = clip(0.40 + 0.14 * strength + 0.04 * heat + rng.normal(0, 0.030, TOTAL_ROWS), 0.30, 1.10)
Mn = clip(0.005 + 0.035 * np.maximum(strength - 0.4, 0) + rng.normal(0, 0.008, TOTAL_ROWS), 0.0, 0.20)
Cu = clip(0.004 + 0.020 * np.maximum(strength - 0.7, 0) + 0.010 * rng.random(TOTAL_ROWS) + rng.normal(0, 0.004, TOTAL_ROWS), 0.0, 0.14)
Cr = clip(0.002 + 0.010 * np.maximum(strength - 0.8, 0) + rng.normal(0, 0.003, TOTAL_ROWS), 0.0, 0.05)
Fe = clip(0.08 + 0.02 * rng.random(TOTAL_ROWS) + 0.012 * np.maximum(strength - 0.2, 0) + rng.normal(0, 0.009, TOTAL_ROWS), 0.03, 0.25)
B = clip(0.0010 + 0.0010 * rng.random(TOTAL_ROWS) + rng.normal(0, 0.00025, TOTAL_ROWS), 0.0002, 0.006)
Ca = clip(0.0006 + 0.0018 * rng.random(TOTAL_ROWS) + rng.normal(0, 0.0003, TOTAL_ROWS), 0.0001, 0.008)
Ni = clip(0.0008 + 0.0016 * rng.random(TOTAL_ROWS) + rng.normal(0, 0.00025, TOTAL_ROWS), 0.0, 0.008)
Ti = clip(0.004 + 0.010 * rng.random(TOTAL_ROWS) + rng.normal(0, 0.002, TOTAL_ROWS), 0.0, 0.05)
V = clip(0.002 + 0.010 * rng.random(TOTAL_ROWS) + rng.normal(0, 0.002, TOTAL_ROWS), 0.0, 0.05)
Zn = clip(0.008 + 0.012 * rng.random(TOTAL_ROWS) + 0.010 * np.maximum(strength - 1.0, 0) + rng.normal(0, 0.003, TOTAL_ROWS), 0.0, 0.08)
Zr = clip(0.001 + 0.006 * np.maximum(strength - 0.9, 0) + rng.normal(0, 0.0015, TOTAL_ROWS), 0.0, 0.03)
Al = clip(
    100 - (B + Ca + Cr + Cu + Fe + Mg + Mn + Ni + Si + Ti + V + Zn + Zr)
    - clip(rng.normal(0.14, 0.03, TOTAL_ROWS), 0.06, 0.24),
    97.6,
    99.4
)

signal = (
    220
    + 40 * strength
    + 54 * (Mg - 0.35)
    + 38 * (Si - 0.40)
    + 55 * Cu
    + 28 * Cr
    + 20 * Mn
    + pd.Series(recipes).map(RECIPE_STRENGTH).to_numpy()
    + 6 * batch
    + 4 * process
)
Rp02 = clip(signal + rng.normal(0, 6, TOTAL_ROWS), 210, 380)
Rm = clip(
    Rp02 + 18 + 9 * np.maximum(heat, 0) + 30 * np.maximum(Si - 0.45, 0) + rng.normal(0, 4, TOTAL_ROWS),
    Rp02 + 12,
    400
)

namo_signal = (
    228
    + 26 * strength
    + 42 * (Mg - 0.35)
    + 28 * (Si - 0.40)
    + 22 * Cu
    + 11 * Mn
    + 10 * heat
    - 9 * np.maximum(strength - 1.0, 0)
    + 3 * batch
)
NaMo_Rp02 = clip(namo_signal + np.where(strength < 0.4, 9, 0) + rng.normal(0, 11, TOTAL_ROWS), 200, 360)
NaMo_Rm = clip(NaMo_Rp02 + 16 + 7 * np.maximum(heat, 0) + rng.normal(0, 8, TOTAL_ROWS), NaMo_Rp02 + 8, 395)

base_df = pd.DataFrame({
    "IMS": ims.astype(str),
    "Oven Recipe": recipes.astype(str),
    "Al": np.round(Al, 4),
    "B": np.round(B, 5),
    "Ca": np.round(Ca, 5),
    "Cr": np.round(Cr, 5),
    "Cu": np.round(Cu, 5),
    "Fe": np.round(Fe, 5),
    "Mg": np.round(Mg, 5),
    "Mn": np.round(Mn, 5),
    "Ni": np.round(Ni, 5),
    "Si": np.round(Si, 5),
    "Ti": np.round(Ti, 5),
    "V": np.round(V, 5),
    "Zn": np.round(Zn, 5),
    "Zr": np.round(Zr, 5),
    "Rp02": np.round(Rp02, 1),
    "Rm": np.round(Rm, 1),
})
base_df.index = np.arange(100000, 100000 + len(base_df))

cat_df = pd.DataFrame({
    "IMS": pd.Categorical(base_df["IMS"], categories=IMS_OPTIONS, ordered=True),
    "Oven Recipe": pd.Categorical(base_df["Oven Recipe"], categories=RECIPE_OPTIONS, ordered=True),
}, index=base_df.index)

ohe_df = pd.get_dummies(
    cat_df,
    columns=["IMS", "Oven Recipe"],
    prefix=["IMS", "Oven Recipe"],
    dtype=int
)

namo_df = pd.DataFrame({
    "NaMo Rp02": np.round(NaMo_Rp02, 1),
    "NaMo Rm": np.round(NaMo_Rm, 1),
}, index=base_df.index)

ds1 = pd.concat([base_df, ohe_df], axis=1)
# ds1.drop(columns=['IMS', 'Oven Recipe'], inplace=True)

## 4. Inspect and Save

The 27+2 column layout of DS1 corresponds to dropping 'IMS' and 'Oven Recipe'. After splitting of cross-validation folds, this is done in `reproduce_paper_I.ipynb`

In [ ]:

print(f"Saved DS1 workbook: {DS1_FILE}")
print(f"Saved NaMo workbook: {NAMO_FILE}")
print(f"DS1 shape: {ds1.shape}")
print(f"NaMo shape: {namo_df.shape}")

print("\nDS1 columns:")
print(list(ds1.columns))

display(ds1.head())
display(namo_df.head())

ds1.to_excel(DS1_FILE)
namo_df.to_excel(NAMO_FILE)

Saved DS1 workbook: C:\Users\chrisdoi\OneDrive - NTNU\PhD\Thesis\codex\i\github\data\DS1.xlsx
Saved NaMo workbook: C:\Users\chrisdoi\OneDrive - NTNU\PhD\Thesis\codex\i\github\data\NaMo_syntetic.xlsx
DS1 shape: (9000, 31)
NaMo shape: (9000, 2)

DS1 columns:
['IMS', 'Oven Recipe', 'Al', 'B', 'Ca', 'Cr', 'Cu', 'Fe', 'Mg', 'Mn', 'Ni', 'Si', 'Ti', 'V', 'Zn', 'Zr', 'Rp02', 'Rm', 'IMS_6060', 'IMS_6063', 'IMS_6005 v1', 'IMS_6005 v2', 'IMS_6082 v1', 'IMS_6082 v2', 'IMS_6082 v3', 'Oven Recipe_Oven Recipe 1', 'Oven Recipe_Oven Recipe 2', 'Oven Recipe_Oven Recipe 3', 'Oven Recipe_Oven Recipe 4', 'Oven Recipe_Oven Recipe 5', 'Oven Recipe_Oven Recipe 6']


,IMS,Oven Recipe,Al,B,Ca,Cr,Cu,Fe,Mg,Mn,...,IMS_6005 v2,IMS_6082 v1,IMS_6082 v2,IMS_6082 v3,Oven Recipe_Oven Recipe 1,Oven Recipe_Oven Recipe 2,Oven Recipe_Oven Recipe 3,Oven Recipe_Oven Recipe 4,Oven Recipe_Oven Recipe 5,Oven Recipe_Oven Recipe 6
100000,6005 v1,Oven Recipe 5,98.6590,0.00195,0.00149,0.00132,0.00672,0.09363,0.55723,0.01452,...,0,0,0,0,0,0,0,0,1,0
100001,6060,Oven Recipe 3,98.9326,0.00131,0.00106,0.00755,0.01413,0.10678,0.38619,0.00000,...,0,0,0,0,0,0,1,0,0,0
100002,6082 v3,Oven Recipe 4,98.1183,0.00209,0.00282,0.00964,0.03741,0.11191,0.76316,0.05781,...,0,0,0,1,0,0,0,1,0,0
100003,6082 v3,Oven Recipe 5,98.3005,0.00085,0.00195,0.01496,0.03168,0.12383,0.62773,0.06103,...,0,0,0,1,0,0,0,0,1,0
100004,6005 v2,Oven Recipe 2,98.4711,0.00131,0.00277,0.00729,0.01952,0.11730,0.60540,0.02617,...,1,0,0,0,0,1,0,0,0,0


,NaMo Rp02,NaMo Rm
100000,264.1,279.1
100001,230.5,246.1
100002,308.3,318.1
100003,286.1,305.3
100004,270.7,291.1
